# Polieus | 02 · Agrupamento dos perfis de demanda urbana

O Polieus é uma proposta de tecnologia cidadã da equipe Politech para aproximar moradores e prefeituras. Uma calçada danificada pode impedir o deslocamento de quem usa cadeira de rodas. Na plataforma proposta, o morador registra a ocorrência, informa o local e acompanha o encaminhamento. A equipe municipal consulta as solicitações para compreender as demandas e explicar suas decisões à população.

Na disciplina de Artificial Intelligence & Chatbot, o grupo utiliza registros públicos do SP156, de São Paulo, para investigar duas questões. A primeira trata da composição dos pedidos em cada distrito e mês. A segunda trata do tempo entre a abertura e a finalização administrativa de uma solicitação. As duas análises partem dos mesmos oito serviços de acessibilidade e pavimentação, observados em outubro–dezembro de 2025 e julho–setembro de 2026. Os meses intermediários ficam fora do recorte.

**Pergunta deste notebook.** Quais distritos, em determinados meses, receberam combinações semelhantes de pedidos? Cada observação reúne um distrito e um mês. O algoritmo aprende perfis a partir da participação relativa de oito serviços. Um gestor pode consultar, por exemplo, quais perfis têm participação mais elevada de solicitações de calçada e investigar as circunstâncias locais.

**Origem dos dados.** O notebook 01 seleciona os serviços e produz as tabelas de estudo. A publicação municipal permite consultar as edições de origem. [Solicitações do SP156 no Portal de Dados Abertos de São Paulo](https://dados.prefeitura.sp.gov.br/dataset/dados-do-sp156)

**Cobertura territorial e temática.** A plataforma recebe relatos de municípios paulistas. Este estudo utiliza os distritos da cidade de São Paulo. Distrito é uma divisão da capital. Aos seis serviços iniciais de acessibilidade e pavimentação somam-se limpeza de bueiros e manutenção de pontos de ônibus, que permitem estudar drenagem da via e condições de circulação. A cobertura representa os pedidos ao SP156 nos períodos selecionados.

### Roteiro de envio manual no Google Colab

| Etapa | Envie pelo painel Arquivos desta sessão | O que será produzido |
|---|---|---|
| 01 · Preparar os dados | `arquivo-final-4-trim-25.csv` e `arquivo-final-3-trim-26.csv` | Dois CSVs para modelagem e uma tabela de cobertura |
| 02 · Agrupar perfis | Somente `02_perfis_por_distrito_e_mes.csv` | Modelo de agrupamento e ficha técnica |
| 03 · Estimar dias | Somente `03_solicitacoes_com_duracao_em_dias.csv` | Modelo de regressão e ficha técnica |

Para reproduzir o estudo desde a origem, abra o notebook 01 pelo menu **Arquivo → Fazer upload de notebook** do Colab. Baixe as duas edições pelos links da primeira seção do 01, mantenha os nomes originais e arraste os CSVs para a pasta principal do painel **Arquivos**. Aguarde o fim do envio e clique em **Executar tudo**. Depois, abra `dados_polieus` e use **⋮ → Fazer download** para guardar as tabelas preparadas no computador.

Abra o 02 ou o 03 em sua própria sessão e arraste apenas o CSV indicado na linha correspondente. O ZIP já inclui esses dois CSVs. Quem quiser reproduzir diretamente um modelo pode usar a cópia incluída. A tabela `01_cobertura_por_servico_e_trimestre.csv` documenta o número de pedidos e a disponibilidade de duração por serviço. Guarde-a para interpretar os resultados. Em uma sessão nova, repita o envio do arquivo correspondente àquela etapa.

### Por que há três notebooks e dois modelos?

A prefeitura publica os chamados em arquivos trimestrais. Para estudar os perfis, precisamos de uma linha por distrito e mês. Para estudar os dias de atendimento, precisamos de uma linha por solicitação com duração conhecida. O notebook 01 transforma as mesmas fontes nessas duas tabelas, documenta as exclusões e permite conferir os totais antes de treinar. Dividir o estudo em três arquivos é uma escolha de organização da equipe. Os modelos aprendem nos notebooks 02 e 03.

O gestor quer saber quais territórios apresentam composições de pedidos semelhantes. Como o histórico não fornece grupos previamente identificados, usamos agrupamento, uma forma de aprendizagem não supervisionada. A segunda pergunta pede um número de dias. Como temos exemplos passados com abertura e finalização, usamos regressão, uma forma de aprendizagem supervisionada. Uma classificação exigiria definir uma categoria-alvo, como uma prioridade validada, e responderia a outra pergunta.

## 1. Primeira execução no Google Colab

1. Baixe o ZIP da entrega. No computador, extraia o ZIP e localize **`dados_polieus/02_perfis_por_distrito_e_mes.csv`**. Se você executou o notebook 01, também pode obter essa tabela pelo painel **Arquivos → dados_polieus → ⋮ → Fazer download**.
2. Abra o [Google Colab](https://colab.research.google.com/) e entre com sua conta Google. Na janela de abertura, escolha **Upload** e selecione o arquivo `.ipynb`. Se já estiver em outro notebook, use **Arquivo → Fazer upload de notebook**. Para guardar suas alterações, salve uma cópia no Drive.
3. Clique em **Conectar**, se solicitado. Use o ambiente padrão Python com CPU. Clique no ícone de **pasta** à esquerda. Arraste **`02_perfis_por_distrito_e_mes.csv`** do computador para o painel de arquivos, ao lado de `sample_data`. Aguarde até aparecer o nome completo. A célula de leitura também aceita a subpasta `dados_polieus`.
4. No menu **Ambiente de execução**, selecione **Executar tudo**. Cada caixa de código é uma célula. O resultado aparece logo abaixo dela. Se preferir estudar por partes, use o triângulo à esquerda de cada célula, de cima para baixo. Se surgir uma mensagem sobre o CSV, confira o nome e o local do upload e volte a executar a partir da leitura.
5. Ao encontrar a mensagem final de exportação verificada, abra a pasta **`modelos_polieus`** no painel lateral. No menu **⋮** de cada arquivo, escolha **Fazer download** para guardar o modelo e a ficha técnica antes de encerrar a sessão. Se o painel ainda não mostrar os arquivos, use o botão de atualização.

A entrada deste notebook é a tabela já preparada. Os CSVs municipais pesados são utilizados somente no notebook 01. Em uma nova sessão do Colab, envie novamente a tabela correspondente.



## 2. Leitura e dicionário dos dados

O nome do distrito e o mês identificam a observação. O campo `total_solicitacoes` informa quantos pedidos sustentam o perfil. Cada coluna iniciada por `proporcao_` representa a fração dos pedidos daquele serviço no total da linha. Um valor de 0,25 corresponde a um quarto das solicitações. As oito proporções somam 1. Dois distritos com volumes diferentes podem apresentar a mesma composição.

### Os oito serviços utilizados

A seleção usa o nome registrado na fonte. Os exemplos ilustram situações de uso. Cada chamado entra uma vez na contagem de seu serviço.

1. **Tapa-buraco.** Pedido de reparo localizado no pavimento de uma via. Um exemplo é um buraco na pista de tráfego comum. O SP156 distingue os pedidos associados a corredores e faixas exclusivas de ônibus.

2. **Tapa-buraco em faixa exclusiva ou corredor de ônibus.** Pedido de reparo no pavimento de uma faixa exclusiva ou corredor de ônibus. A localização operacional distingue essa classe do tapa-buraco em via comum.

3. **Acessibilidade - solicitar avaliação de obstáculo na calçada.** Solicitação de avaliação de uma barreira no passeio que dificulta a circulação. O registro representa um pedido de vistoria ou avaliação. A remoção efetiva do obstáculo exige confirmação posterior.

4. **Calçada pública - solicitar manutenção.** Pedido de manutenção de uma calçada identificada como pública. Um exemplo é um trecho de passeio público com piso danificado que dificulta a passagem a pé.

5. **Calçada particular - denunciar calçada danificada ou inexistente.** Denúncia de passeio particular danificado ou inexistente. O trâmite pode envolver fiscalização. A duração administrativa precisa ser interpretada conforme esse tipo de atendimento.

6. **Guias para travessia de pedestres - Solicitar rebaixamento.** Pedido de rebaixamento de guia em travessia de pedestres, por exemplo para facilitar a passagem de cadeira de rodas entre a calçada e a pista.

7. **Solicitar limpeza de bueiros, boca de lobo e poços de visita.** Solicitação de limpeza de estruturas da drenagem urbana, como uma boca de lobo obstruída. A categoria permite observar demandas de escoamento de água em vias de circulação.

8. **Ponto de ônibus – Solicitar manutenção, conserto ou limpeza de equipamento existente.** Pedido de manutenção, conserto ou limpeza de equipamento já existente em ponto de ônibus. Um exemplo é um abrigo danificado. Reclamações sobre intervalos de linhas e pedidos de implantação têm outras categorias.

In [ ]:
from pathlib import Path
import json
import platform
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import FunctionTransformer
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

In [ ]:
arquivo = Path("02_perfis_por_distrito_e_mes.csv")
if not arquivo.is_file():
    arquivo = Path("dados_polieus") / arquivo
if not arquivo.is_file():
    raise FileNotFoundError("Envie 02_perfis_por_distrito_e_mes.csv pelo painel Arquivos do Colab e execute novamente.")
dados = pd.read_csv(arquivo)
print("Linhas e colunas:", dados.shape)
display(dados.head(3))

servicos = [
    "Tapa-buraco", "Tapa-buraco em faixa exclusiva ou corredor de ônibus",
    "Acessibilidade - solicitar avaliação de obstáculo na calçada", "Calçada pública - solicitar manutenção",
    "Calçada particular - denunciar calçada danificada ou inexistente",
    "Guias para travessia de pedestres - Solicitar rebaixamento",
]
servicos.extend(['Solicitar limpeza de bueiros, boca de lobo e poços de visita', 'Ponto de ônibus – Solicitar manutenção, conserto ou limpeza de equipamento existente'])
atributos = ["proporcao_" + nome for nome in servicos]
campos = ["distrito", "mes", "total_solicitacoes"] + atributos
if not set(campos).issubset(dados):
    raise ValueError("O CSV precisa conter os nomes de coluna gerados no notebook 01.")
if dados[campos].isna().any().any() or dados.duplicated(["distrito", "mes"]).any():
    raise ValueError("Confira valores ausentes e a unicidade de cada par distrito e mês.")
X = dados[atributos]
if not ((X >= 0) & (X <= 1)).all().all() or not np.allclose(X.sum(axis=1), 1):
    raise ValueError("Cada proporção deve estar entre 0 e 1 e as oito devem somar 1.")
volume = dados["total_solicitacoes"]
if not ((volume >= 1) & (volume % 1 == 0)).all():
    raise ValueError("Cada perfil exige um número inteiro e positivo de solicitações.")
dados["mes"] = pd.to_datetime(dados["mes"], format="%Y-%m").dt.to_period("M")
display(dados.groupby("mes").agg(perfis=("distrito", "size"), pedidos=("total_solicitacoes", "sum")))

## 3. Separação temporal e exploração dos perfis de treino

Usamos o quarto trimestre de 2025 para aprender os centros dos grupos, julho de 2026 para escolher a configuração e agosto–setembro de 2026 para avaliá-la. O mesmo distrito pode aparecer em meses distintos, como acontecerá ao acompanhar um município ao longo do tempo. Essa avaliação investiga a aplicação a **meses novos na mesma cidade**.

O gráfico resume a composição do treino. A média simples atribui um voto a cada distrito-mês. A média ponderada por volume atribui um voto a cada solicitação. Observe se as duas leituras contam a mesma história. Os perfis com poucos pedidos são mantidos e examinados novamente na análise de sensibilidade.

In [ ]:
treino = dados.loc[dados["mes"].between("2025-10", "2025-12")].copy()
validacao = dados.loc[dados["mes"] == "2026-07"].copy()
teste = dados.loc[dados["mes"].between("2026-08", "2026-09")].copy()
if min(len(treino), len(validacao), len(teste)) == 0 or len(treino)+len(validacao)+len(teste) != len(dados):
    raise ValueError("Confira os meses do recorte preparado no notebook 01.")

resumo = pd.DataFrame({
    "Cada distrito-mês": treino[atributos].mean(),
    "Cada solicitação": np.average(treino[atributos], axis=0, weights=treino["total_solicitacoes"]),
})
resumo.index = ["Tapa-buraco", "Corredor de ônibus", "Obstáculo", "Calçada pública", "Calçada particular", "Guias", "Bueiros", "Pontos de ônibus"]
(resumo * 100).plot.barh(figsize=(8, 4), xlabel="Participação no treino (%)")
plt.tight_layout()
plt.show()
print("Perfis de treino, validação e teste:", len(treino), len(validacao), len(teste))
print("Perfis com menos de 5 pedidos no treino:", (treino.total_solicitacoes < 5).sum())
display(treino.total_solicitacoes.describe())

## 4. Como o agrupamento aprende e como escolhemos o modelo

O **K-Means** procura centros que representem as observações do treino e atribui cada perfil ao centro mais próximo. O número do grupo identifica o centro. Seu valor numérico é arbitrário. Como o conjunto não tem rótulos de “grupo correto”, a avaliação considera a separação geométrica, o tamanho e a interpretação dos grupos.

Experimentamos de dois a cinco centros e duas representações dos mesmos oito atributos. A primeira usa as proporções originais. A segunda usa a raiz quadrada de cada proporção, o que dá mais peso a diferenças entre participações pequenas. Distrito, mês e volume ficam disponíveis para a interpretação. Atribuímos o mesmo peso a cada distrito-mês durante o ajuste.

Em julho, medimos a **silhueta** (mais próxima de 1 indica maior separação) e o índice **Davies–Bouldin** (menor indica grupos mais compactos e separados). Para comparar as duas representações, avaliamos todas as atribuições no mesmo espaço das raízes quadradas. Antes da seleção, exigimos pelo menos cinco exemplos por grupo no treino e na validação como proteção contra segmentos muito pequenos. Entre os candidatos admissíveis, prevalece a maior silhueta de validação, com desempate pelo menor número de centros. A escolha desse limite é metodológica.

### Como ler os conceitos do agrupamento

**Distrito-mês** é a unidade de observação. Água Rasa em agosto e Água Rasa em setembro são duas linhas. Cada uma descreve um distrito da capital durante um mês de abertura. Se houver 20 solicitações e cinco forem de limpeza de bueiros, a participação desse serviço será 5 ÷ 20 = 25%. O denominador inclui os oito serviços selecionados.

**Agrupamento territorial** reúne linhas que apresentam composições semelhantes. Distritos distantes podem aparecer no mesmo grupo. **K-Means** é o algoritmo utilizado. A letra K indica a quantidade de grupos experimentada. O algoritmo atribui cada perfil ao centro mais próximo, recalcula os centros a partir dos perfis atribuídos e repete esse processo até estabilizar ou atingir o limite de iterações. Cada centro resume um perfil típico. Grupo 0 e grupo 1 são identificadores, sem ordem de urgência.

**Silhueta** compara a distância de uma observação aos membros do próprio grupo com sua distância aos membros do grupo vizinho mais próximo, em média. A escala vai de −1 a 1. Valores próximos de 1 indicam separação clara; perto de zero, os grupos se sobrepõem; um valor negativo sugere maior proximidade com outro grupo. Neste teste, a média de 0,283 indica separação limitada.

**ARI**, sigla de *Adjusted Rand Index* (índice de Rand ajustado), compara como duas divisões agrupam os mesmos pares de observações, com ajuste para concordância esperada ao acaso. Vale 1 para divisões equivalentes, mesmo que os números dos grupos sejam trocados. Zero corresponde à concordância esperada ao acaso; valores negativos também são possíveis. O ARI de 0,913 mostra grande concordância com o agrupamento que usa somente a participação de tapa-buraco. Esse valor não representa porcentagem de acertos.

**Estabilidade** avalia se a divisão muda muito quando alteramos a amostra de treino. **Davies–Bouldin** compara a dispersão interna dos grupos com a separação entre os centros. Valores menores favorecem grupos mais compactos e separados na representação avaliada. Nenhuma dessas medidas determina qual distrito deve receber uma obra primeiro.

**Como ler o placar.** Cada linha testa uma combinação. k é o número de grupos. A coluna raiz indica se foi aplicada a raiz quadrada às proporções. O menor grupo precisa ter ao menos cinco perfis no treino e na validação. Entre as configurações elegíveis, escolhemos a maior silhueta e, em caso de empate, o menor k. Davies–Bouldin complementa a leitura: valores menores indicam grupos mais compactos e separados segundo essa medida. A planilha na pasta consulta_excel apresenta as proporções como porcentagens para consulta. Envie ao Colab o CSV original da pasta dados_polieus.

In [ ]:
def construir_modelo(k, raiz):
    transformacao = FunctionTransformer(np.sqrt) if raiz else "passthrough"
    return Pipeline([
        ("composicao", transformacao),
        ("centros", KMeans(n_clusters=k, n_init=20, random_state=42)),
    ])

def qualidade(amostra, grupos):
    pontos = np.sqrt(amostra[atributos])
    if not 1 < len(np.unique(grupos)) < len(amostra):
        return np.nan, np.nan
    return silhouette_score(pontos, grupos), davies_bouldin_score(pontos, grupos)

comparacoes = []
for k in range(2, 6):
    for raiz in [False, True]:
        candidato = construir_modelo(k, raiz).fit(treino[atributos])
        g_treino = candidato.predict(treino[atributos])
        g_val = candidato.predict(validacao[atributos])
        sil, db = qualidade(validacao, g_val)
        menor = min(np.bincount(g_treino, minlength=k).min(), np.bincount(g_val, minlength=k).min())
        comparacoes.append(dict(k=k, raiz=raiz, silhueta=sil, davies_bouldin=db, menor_grupo=menor))
placar = pd.DataFrame(comparacoes).sort_values(["silhueta", "k"], ascending=[False, True])
display(placar.round(3))
aptos = placar.loc[placar.menor_grupo >= 5]
if aptos.empty:
    raise ValueError("Nenhuma configuração atingiu o suporte mínimo. Revise a composição antes de segmentar.")
escolha = aptos.iloc[0]
k = int(escolha.k)
modelo = construir_modelo(k, bool(escolha.raiz)).fit(treino[atributos])
print("Centros escolhidos:", k, "| transformação por raiz:", bool(escolha.raiz))

## 5. Os grupos acrescentam informação ao percentual de tapa-buraco?

O serviço de tapa-buraco domina o recorte. Construímos uma referência que usa somente essa proporção e o mesmo número de grupos. Tanto a referência quanto o modelo principal aprendem os seus centros nos mesmos meses. Para cada grupo, calculamos a composição média observada no treino e usamos essa média para reconstruir a composição de cada perfil de teste.

A tabela mede o erro absoluto médio de reconstrução em **pontos percentuais**. A segunda coluna de erro considera apenas os outros sete serviços. Uma redução nessa coluna sugere que o modelo completo resume melhor as categorias de menor volume. O **índice de Rand ajustado (ARI)** compara duas divisões dos mesmos registros, mesmo quando a numeração dos grupos muda. Um valor de 1 indica concordância completa e um valor próximo de 0 indica concordância próxima ao acaso. A interpretação municipal depende também dos exemplos de cada perfil.

In [ ]:
ref = KMeans(n_clusters=k, n_init=20, random_state=42).fit(treino[[atributos[0]]])
grupos = modelo.predict(teste[atributos])
ref_teste = ref.predict(teste[[atributos[0]]])
g_treino = modelo.predict(treino[atributos])

def erro_perfil(grupos_ajuste, grupos_novos):
    medias = treino[atributos].groupby(grupos_ajuste).mean()
    previstos = medias.loc[grupos_novos].to_numpy()
    erros = np.abs(teste[atributos].to_numpy() - previstos) * 100
    return erros.mean(), erros[:, 1:].mean()

linhas = []
for rotulo, origem, destino in [
    ("Oito serviços", g_treino, grupos), ("Somente tapa-buraco", ref.labels_, ref_teste)
]:
    geral, outros = erro_perfil(origem, destino)
    sil, db = qualidade(teste, destino)
    linhas.append([rotulo, geral, outros, sil, db])
avaliacao = pd.DataFrame(linhas, columns=["Representação", "Erro geral (pp)", "Erro outros (pp)", "Silhueta", "Davies–Bouldin"])
display(avaliacao.round(3))
ari_ref = adjusted_rand_score(grupos, ref_teste)
print("Concordância com a referência (ARI):", round(ari_ref, 3))

## 6. Leitura territorial e estabilidade

A primeira tabela descreve a participação média dos serviços em cada grupo aprendido. A segunda conta os distritos-mês de teste atribuídos a cada centro. Escolha uma linha da tabela de médias e procure o que mais a distingue das outras. Mesmo quando tapa-buraco predomina em todos os grupos, um grupo pode se distinguir pela participação de manutenção de calçadas.

Repetimos o ajuste retirando do treino perfis com menos de cinco pedidos e, separadamente, usando cinco subamostras de 80% do treino. Comparamos as atribuições nos mesmos meses reservados. Se o grupo mudar com frequência, a segmentação exige cautela. Os pedidos expressam tanto necessidades urbanas quanto diferenças de acesso ao canal SP156. Decisões de prioridade exigem revisão humana e informações sobre as barreiras encontradas pelos moradores.

In [ ]:
medias = treino[atributos].groupby(g_treino).mean()
perfis_pct = (100 * medias).round(2)
perfis_pct.columns = resumo.index
perfis_pct.index.name = "grupo"
display(perfis_pct)
display(pd.crosstab(teste["mes"], grupos).reindex(columns=range(k), fill_value=0))
for mes, amostra in teste.groupby("mes"):
    sil, db = qualidade(amostra, modelo.predict(amostra[atributos]))
    print(mes, "| silhueta:", round(sil, 3), "| Davies–Bouldin:", round(db, 3))
estavel = treino.loc[treino.total_solicitacoes >= 5]
alternativo = clone(modelo).fit(estavel[atributos])
ari_volume = adjusted_rand_score(grupos, alternativo.predict(teste[atributos]))
print("Perfis retirados do treino:", len(treino) - len(estavel), "| ARI:", round(ari_volume, 3))
aris = []
for semente in range(5):
    amostra = treino.sample(frac=0.8, random_state=semente)
    replica = clone(modelo).fit(amostra[atributos])
    aris.append(adjusted_rand_score(grupos, replica.predict(teste[atributos])))
print("ARI nas cinco subamostras:", np.round(aris, 3))
display(teste[["distrito", "mes", "total_solicitacoes"]].assign(grupo=grupos).groupby("grupo").head(2))

### Interpretação dos perfis

Compare a participação de bueiros, tapa-buraco e dos demais serviços em cada centro. A concordância alta com a referência indica que uma parte importante da separação já pode ser explicada pela participação de tapa-buraco. Use a descrição como ponto de partida para investigação local.

No teste de referência, a silhueta foi 0,283 e o ARI frente à referência foi 0,913. A limpeza de bueiros representou 14,5% dos pedidos no centro mais concentrado em tapa-buraco e 31,1% no outro.

**Leitura da execução de referência.** O K-Means formou dois grupos. No treino, a limpeza de bueiros representou em média 14,5% dos pedidos do grupo 0 e 31,1% dos pedidos do grupo 1. No teste, o ARI de 0,913 mostra que a divisão se parece bastante com a obtida usando apenas a proporção de tapa-buraco. A silhueta de 0,283 sugere que há perfis próximos da fronteira entre os grupos. Os números dos grupos são identificadores e não expressam urgência.

## 7. Exportação e uso do perfil na API Flask

Guardamos o mesmo modelo avaliado, com a transformação e os centros aprendidos. A API Flask recebe as oito proporções de um distrito-mês, com os nomes da tabela de entrada. Deve verificar se são finitas, estão entre 0 e 1 e somam 1. Se receber contagens, o total precisa ser positivo para calcular as proporções. O identificador de grupo acompanha a descrição das médias observadas e o volume que sustentou a entrada.

A célula grava o modelo e uma ficha JSON com a ordem dos campos, os perfis e a versão das bibliotecas. Depois reabre o modelo e repete uma atribuição do teste. A igualdade demonstra que a serialização preservou a resposta nesse exemplo. A numeração dos grupos pertence a esta versão do modelo.

In [ ]:
pasta = Path("modelos_polieus")
pasta.mkdir(exist_ok=True)
arquivo_modelo = pasta / "modelo_perfis_territoriais.joblib"
joblib.dump(modelo, arquivo_modelo, compress=3)
recuperado = joblib.load(arquivo_modelo)
exemplo = teste[atributos].iloc[[0]]
antes = modelo.predict(exemplo)
depois = recuperado.predict(exemplo)
np.testing.assert_array_equal(antes, depois)
ficha = {
    "modelo": "polieus_agrupamento",
    "python": platform.python_version(),
    "scikit_learn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "joblib": joblib.__version__,
    "entradas": atributos, "k": k, "raiz": bool(escolha.raiz),
    "perfis_medios_percentuais": {int(g): linha.to_dict() for g, linha in perfis_pct.iterrows()},
    "periodo_ajuste": "2025-10 a 2025-12", "periodo_teste": "2026-08 a 2026-09",
    "metricas_teste": avaliacao.to_dict("records"), "ari_referencia": float(ari_ref),
}
(pasta / "ficha_perfis_territoriais.json").write_text(json.dumps(ficha, ensure_ascii=False, indent=2), encoding="utf-8")
display(exemplo)
print("Grupo atribuído ao exemplo real:", depois[0])
print("Exportação verificada.", arquivo_modelo)